In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time, math
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config, features
import numpy as np, pandas as pd
print('ready:', os.getcwd())

Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# 07_remedies
# E1: which published remedies close the SHC gap, and what does each require?
#
# Preregistered success criterion:
#   identify at least one shift mechanism under which NO label-free remedy
#   restores focal-class coverage within 2 percentage points of nominal at a
#   mean set size below half the label space.
# =============================================================================
INFO_REQUIREMENTS = pd.DataFrame([
    dict(remedy='SHC (no remedy)',      unlabelled_target='no',  target_labels='no',
         streaming='no',  deployable='yes'),
    dict(remedy='Weighted conformal',   unlabelled_target='yes', target_labels='no',
         streaming='no',  deployable='yes'),
    dict(remedy='Entropy adaptation',   unlabelled_target='yes', target_labels='no',
         streaming='no',  deployable='yes'),
    dict(remedy='Adaptive conformal',   unlabelled_target='no',  target_labels='DELAYED',
         streaming='yes', deployable='only with label feedback'),
    dict(remedy='NexCP recency',        unlabelled_target='no',  target_labels='no',
         streaming='yes', deployable='requires temporal order'),
    dict(remedy='TSC (oracle)',         unlabelled_target='yes', target_labels='yes',
         streaming='no',  deployable='no'),
])
print(INFO_REQUIREMENTS.to_string(index=False))

ALPHA = config.ALPHA_PRIMARY
NOMINAL = 1 - ALPHA
TOL = 0.02                      # within 2 points of nominal
SET_CAP = len(config.CANONICAL_CLASSES) / 2.0
print(f'\nrestoration bar: coverage >= {NOMINAL-TOL:.3f} AND mean set size < {SET_CAP}')
print('NexCP is NOT APPLICABLE to NSL-KDD: the dataset carries no temporal order.')
print('It is deferred to UGR-16.')

            remedy unlabelled_target target_labels streaming               deployable
   SHC (no remedy)                no            no        no                      yes
Weighted conformal               yes            no        no                      yes
Entropy adaptation               yes            no        no                      yes
Adaptive conformal                no       DELAYED       yes only with label feedback
     NexCP recency                no            no       yes  requires temporal order
      TSC (oracle)               yes           yes        no                       no

restoration bar: coverage >= 0.930 AND mean set size < 2.5
NexCP is NOT APPLICABLE to NSL-KDD: the dataset carries no temporal order.
It is deferred to UGR-16.


In [3]:
# =============================================================================
# Cell 3 - load partitions, ladder, cached probabilities
# =============================================================================
nsl_train = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_train.parquet').reset_index(drop=True)
nsl_test  = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_test.parquet').reset_index(drop=True)
part = pd.read_parquet(config.PROC_DIR / 'nslkdd_source_partition_labels.parquet')
nsl_train = nsl_train.assign(partition=part['partition'].values)

CLASSES = config.CANONICAL_CLASSES; K = len(CLASSES)
c2i = {c: i for i, c in enumerate(CLASSES)}
FOCAL = json.loads((config.REPORTS_DIR / 'focal_class_record.json').read_text())['focal_class']
FOCAL_I = c2i[FOCAL]

S_pool    = nsl_train[nsl_train.partition == 'source_cal_pool']
D_probcal = nsl_train[nsl_train.partition == 'probcal']
y_sp = S_pool['label'].map(c2i).to_numpy()
y_te = nsl_test['label'].map(c2i).to_numpy()

COLS = features.feature_cols(nsl_train)
CATS = features.fit_categories(nsl_train, nsl_test)
X_sp = features.encode(S_pool, COLS, CATS)
X_pc = features.encode(D_probcal, COLS, CATS)
X_te = features.encode(nsl_test, COLS, CATS)

assign = pd.read_parquet(config.PROC_DIR / 'nslkdd_ladder_assignments.parquet')
IDX = {(r, j, role): g['test_idx'].to_numpy()
       for (r, j, role), g in assign.groupby(['rung','realization','role'])}
INSTANCES = sorted({(r, j) for (r, j, _) in IDX})
MODELS = sorted(glob.glob(str(config.PROC_DIR / 'probs_*.npz')))
print(f'{len(INSTANCES)} instances | {len(MODELS)} models | focal {FOCAL}')

100 instances | 30 models | focal R2L


In [4]:
# =============================================================================
# Cell 4 - scores and quantiles (identical spec to notebook 05)
# =============================================================================
from numpy.random import Generator, Philox

def draw_U(n, k, seed, stream):
    return Generator(Philox(key=int(seed), counter=int(stream))).random((n, k))

def aps_scores(P, U):
    gt = P[:, None, :] > P[:, :, None]
    return np.einsum('iyj,ij->iy', gt.astype(P.dtype), P) + U * P

def conformal_q(scores, alpha):
    n = len(scores)
    if n == 0: return np.inf
    k = math.ceil((n + 1) * (1 - alpha))
    return np.inf if k > n else float(np.sort(scores)[k - 1])

def weighted_q(scores, weights, alpha):
    """Weighted empirical quantile, Tibshirani et al. 2019."""
    if len(scores) == 0: return np.inf
    o = np.argsort(scores); s, w = scores[o], weights[o]
    cw = np.cumsum(w) / w.sum()
    i = int(np.searchsorted(cw, 1 - alpha))
    return float(s[min(i, len(s) - 1)])

def summarise(inset, y_ev):
    cov = inset[np.arange(len(y_ev)), y_ev]
    size = inset.sum(1)
    m = y_ev == FOCAL_I
    return dict(cov_marg=float(cov.mean()),
                cov_focal=float(cov[m].mean()) if m.any() else np.nan,
                set_mean=float(size.mean()),
                set_focal=float(size[m].mean()) if m.any() else np.nan,
                empty=float((size == 0).mean()), full=float((size == K).mean()))

print('score and quantile helpers ready')

score and quantile helpers ready


In [5]:
# =============================================================================
# Cell 5 - weighted conformal: density-ratio weights from UNLABELLED target
#
# Binding constraint (preregistration section 9): the domain classifier that
# supplies these weights must be fitted on data DISJOINT from the one used for
# S_cov. S_cov used S_pool and D_eval. This one uses D_probcal versus D_eval,
# so the two never share a fitting sample.
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier

def density_ratio_weights(X_src_fit, X_tgt_fit, X_apply, seed=0):
    n0, n1 = len(X_src_fit), len(X_tgt_fit)
    X = np.vstack([X_src_fit, X_tgt_fit])
    y = np.r_[np.zeros(n0), np.ones(n1)]
    clf = HistGradientBoostingClassifier(max_iter=80, max_depth=6, random_state=seed)
    clf.fit(X, y)
    p = np.clip(clf.predict_proba(X_apply)[:, 1], 1e-6, 1 - 1e-6)
    w = (p / (1 - p)) * (n0 / n1)
    return np.clip(w, 1e-3, 1e3)      # clipped: unbounded ratios destabilise the quantile

t0 = time.time()
WEIGHTS = {}
for i, (rung, real) in enumerate(INSTANCES):
    e = IDX[(rung, real, 'eval')]
    WEIGHTS[(rung, real)] = density_ratio_weights(X_pc, X_te[e], X_sp, seed=i)
    if (i + 1) % 25 == 0: print(f'  {i+1}/{len(INSTANCES)}  {time.time()-t0:.0f}s')
print(f'density-ratio weights for {len(WEIGHTS)} instances in {time.time()-t0:.0f}s')
w0 = WEIGHTS[INSTANCES[0]]
print(f'example weights: median {np.median(w0):.3f}  p95 {np.quantile(w0,0.95):.3f}  '
      f'max {w0.max():.3f}')

  25/100  24s
  50/100  49s
  75/100  75s
  100/100  100s
density-ratio weights for 100 instances in 100s
example weights: median 0.440  p95 1.764  max 22.312


In [6]:
# =============================================================================
# Cell 6 - entropy adaptation, in the spirit of ECP/EACP (Kasa et al. 2024)
#
# NOT an exact reimplementation. Temperature is fitted so that mean predictive
# entropy on the UNLABELLED target matches the source calibration pool, i.e.
# the target is mapped onto the distribution the quantile was calibrated for.
# Labelled as an entropy-matching adaptation in the paper, not as ECP itself.
# =============================================================================
from scipy.optimize import minimize_scalar

def entropy(P, eps=1e-12):
    P = np.clip(P, eps, 1.0)
    return float(np.mean(-(P * np.log(P)).sum(1)))

def temp_scale(P, T, eps=1e-12):
    Q = np.clip(P, eps, 1.0) ** (1.0 / T)
    return Q / Q.sum(1, keepdims=True)

def fit_temperature(P_target, H_source):
    f = lambda logT: (entropy(temp_scale(P_target, np.exp(logT))) - H_source) ** 2
    r = minimize_scalar(f, bounds=(np.log(0.05), np.log(20.0)), method='bounded')
    return float(np.exp(r.x))

print('entropy adaptation ready')

entropy adaptation ready


In [7]:
# =============================================================================
# Cell 7 - adaptive conformal inference (Gibbs and Candes 2021)
#   alpha_{t+1} = alpha_t + gamma * (alpha - err_t)
# err_t requires knowing whether the true label fell in the set, i.e. LABELS.
# Run twice: with label feedback, and without, to make the requirement concrete.
# =============================================================================
GAMMA = 0.01

def aci_stream(S_ev, y_ev, S_cal, y_cal, alpha0, mondrian=True, feedback=True, seed=0):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(y_ev))
    a = alpha0
    covered = np.zeros(len(y_ev), dtype=bool); sizes = np.zeros(len(y_ev))
    for t in order:
        if mondrian:
            q = np.array([conformal_q(S_cal[y_cal == c, c], np.clip(a, 1e-4, 0.999))
                          for c in range(K)])
            ins = S_ev[t] <= q
        else:
            q = conformal_q(S_cal[np.arange(len(y_cal)), y_cal], np.clip(a, 1e-4, 0.999))
            ins = S_ev[t] <= q
        covered[t] = ins[y_ev[t]]; sizes[t] = ins.sum()
        if feedback:
            a = a + GAMMA * (alpha0 - (0.0 if covered[t] else 1.0))
        # without feedback err_t is unknown, so alpha cannot update: a stays fixed
    return covered, sizes

print('ACI ready; gamma =', GAMMA)

ACI ready; gamma = 0.01


In [8]:
# =============================================================================
# Cell 8 - main sweep
# Runs at the primary specification only: alpha 0.05, APS, Mondrian, focal R2L.
# ACI is evaluated on a subsample of instances because it is sequential.
# =============================================================================
ACI_INSTANCES = [inst for inst in INSTANCES if inst[1] < 3]     # 3 realizations per rung
rows = []
t0 = time.time()

for mi, mfile in enumerate(MODELS):
    arch, seed = Path(mfile).stem.replace('probs_', '').rsplit('_s', 1); seed = int(seed)
    z = np.load(mfile)
    P_te = z['target'].astype(np.float64); P_sp = z['S_pool'].astype(np.float64)
    U_te = draw_U(len(P_te), K, seed, 1); U_sp = draw_U(len(P_sp), K, seed, 2)
    S_te = aps_scores(P_te, U_te); S_sp = aps_scores(P_sp, U_sp)
    H_src = entropy(P_sp)

    for rung, real in INSTANCES:
        e = IDX[(rung, real, 'eval')]; t = IDX[(rung, real, 'tcal')]
        ye = y_te[e]
        base = dict(rung=rung, realization=real, arch=arch, seed=seed)

        # --- SHC baseline -------------------------------------------------
        q = np.array([conformal_q(S_sp[y_sp == c, c], ALPHA) for c in range(K)])
        rows.append({**base, 'remedy': 'SHC', **summarise(S_te[e] <= q, ye)})

        # --- TSC oracle ---------------------------------------------------
        q = np.array([conformal_q(S_te[t][y_te[t] == c, c], ALPHA) for c in range(K)])
        rows.append({**base, 'remedy': 'TSC_oracle', **summarise(S_te[e] <= q, ye)})

        # --- weighted conformal -------------------------------------------
        w = WEIGHTS[(rung, real)]
        q = np.array([weighted_q(S_sp[y_sp == c, c], w[y_sp == c], ALPHA) for c in range(K)])
        rows.append({**base, 'remedy': 'WeightedCP', **summarise(S_te[e] <= q, ye)})

        # --- entropy adaptation --------------------------------------------
        T = fit_temperature(P_te[e], H_src)
        S_adapt = aps_scores(temp_scale(P_te[e], T), U_te[e])
        q = np.array([conformal_q(S_sp[y_sp == c, c], ALPHA) for c in range(K)])
        r = summarise(S_adapt <= q, ye); r['temperature'] = T
        rows.append({**base, 'remedy': 'EntropyAdapt', **r})

    for rung, real in ACI_INSTANCES:
        e = IDX[(rung, real, 'eval')]; ye = y_te[e]
        for fb, name in [(True, 'ACI_with_labels'), (False, 'ACI_no_labels')]:
            cvd, sz = aci_stream(S_te[e], ye, S_sp, y_sp, ALPHA, feedback=fb, seed=seed)
            m = ye == FOCAL_I
            rows.append(dict(rung=rung, realization=real, arch=arch, seed=seed,
                             remedy=name, cov_marg=float(cvd.mean()),
                             cov_focal=float(cvd[m].mean()) if m.any() else np.nan,
                             set_mean=float(sz.mean()),
                             set_focal=float(sz[m].mean()) if m.any() else np.nan,
                             empty=float((sz == 0).mean()), full=float((sz == K).mean())))

    print(f'  [{mi+1}/{len(MODELS)}] {arch}_s{seed}  {time.time()-t0:.0f}s')

rem = pd.DataFrame(rows)
print(f'\ndone in {time.time()-t0:.0f}s | {len(rem):,} rows')

  [1/30] mlp_s12345  67s
  [2/30] mlp_s1337  137s
  [3/30] mlp_s2024  205s
  [4/30] mlp_s3407  273s
  [5/30] mlp_s42  340s
  [6/30] mlp_s512  410s
  [7/30] mlp_s6021  477s
  [8/30] mlp_s7  546s
  [9/30] mlp_s88  614s
  [10/30] mlp_s91  682s
  [11/30] rf_s12345  750s
  [12/30] rf_s1337  818s
  [13/30] rf_s2024  888s
  [14/30] rf_s3407  955s
  [15/30] rf_s42  1025s
  [16/30] rf_s512  1092s
  [17/30] rf_s6021  1162s
  [18/30] rf_s7  1231s
  [19/30] rf_s88  1299s
  [20/30] rf_s91  1368s
  [21/30] xgb_s12345  1436s
  [22/30] xgb_s1337  1505s
  [23/30] xgb_s2024  1574s
  [24/30] xgb_s3407  1644s
  [25/30] xgb_s42  1713s
  [26/30] xgb_s512  1781s
  [27/30] xgb_s6021  1850s
  [28/30] xgb_s7  1919s
  [29/30] xgb_s88  1986s
  [30/30] xgb_s91  2055s

done in 2055s | 12,900 rows


In [9]:
# =============================================================================
# Cell 9 - verdict against the preregistered criterion
# =============================================================================
summary = (rem.groupby(['remedy','rung'])[['cov_focal','set_focal','cov_marg','set_mean']]
           .mean().round(4))
print(f'FOCAL CLASS {FOCAL}, alpha {ALPHA}')
print(summary[['cov_focal','set_focal']].unstack('rung').round(3).to_string())
print('\nmarginal coverage')
print(summary['cov_marg'].unstack('rung').round(3).to_string())

LABEL_FREE = ['WeightedCP', 'EntropyAdapt', 'ACI_no_labels']

def restores(g):
    return bool((g['cov_focal'] >= NOMINAL - TOL) and (g['set_focal'] < SET_CAP))

per = (rem.groupby(['remedy','rung'])[['cov_focal','set_focal']].mean().reset_index())
per['restores'] = per.apply(restores, axis=1)

print('\nrestoration by remedy and rung (bar: coverage >= %.3f AND focal set < %.1f)'
      % (NOMINAL - TOL, SET_CAP))
piv = per.pivot(index='remedy', columns='rung', values='restores')
print(piv.to_string())

# preregistered criterion: does there EXIST a rung at which NO label-free
# remedy restores? Not: do all label-free remedies fail everywhere.
lf = per[per.remedy.isin(LABEL_FREE)]
any_lf = lf.groupby('rung')['restores'].any()
print('\nany label-free remedy restores, by rung:')
print(any_lf.to_string())

failed_rungs = [float(r) for r, v in any_lf.items() if not v]
E1_PASS = len(failed_rungs) > 0
print('\nE1 criterion:', 'PASS' if E1_PASS else 'FAIL')
if E1_PASS:
    print(f'  no label-free remedy restores focal coverage at rung(s): {failed_rungs}')
    ok_rungs = [float(r) for r, v in any_lf.items() if v]
    if ok_rungs:
        print(f'  a label-free remedy DOES restore at rung(s): {ok_rungs}')
        print('  => the remedy landscape is mechanism-dependent, which is the finding,')
        print('     not a weakness. Report both halves.')
else:
    print('  a label-free remedy restores at every rung; the deployment-gap claim fails')

res = (per.groupby('remedy')
         .agg(label_free=('remedy', lambda x: x.iloc[0] in LABEL_FREE),
              rungs_restored=('restores','sum'), n_rungs=('restores','size'),
              min_cov_focal=('cov_focal','min'), max_set_focal=('set_focal','max'))
         .reset_index().sort_values('label_free', ascending=False))
print('\n' + res.to_string(index=False))

FOCAL CLASS R2L, alpha 0.05
                cov_focal                             set_focal                            
rung                  0.0    0.2    0.4    0.6    0.8       0.0    0.2    0.4    0.6    0.8
remedy                                                                                     
ACI_no_labels       0.150  0.105  0.089  0.063  0.035     2.121  2.098  2.079  2.060  2.066
ACI_with_labels     0.796  0.948  0.962  0.993  0.992     3.032  3.940  4.205  4.629  4.609
EntropyAdapt        0.087  0.066  0.050  0.034  0.016     1.992  1.980  1.972  1.966  1.960
SHC                 0.143  0.114  0.085  0.058  0.030     2.114  2.120  2.076  2.063  2.045
TSC_oracle          0.954  0.955  0.953  0.953  0.953     3.091  3.663  3.998  4.117  4.678
WeightedCP          0.427  0.396  0.361  0.209  0.164     1.619  1.622  1.551  1.408  1.347

marginal coverage
rung               0.0    0.2    0.4    0.6    0.8
remedy                                            
ACI_no_labels    0.830 

In [10]:
# =============================================================================
# Cell 10 - persist and commit
# =============================================================================
rem.to_parquet(config.PROC_DIR / 'remedies_nslkdd.parquet', index=False)
summary.to_csv(config.REPORTS_DIR / 'remedies_summary_nslkdd.csv')
INFO_REQUIREMENTS.to_csv(config.REPORTS_DIR / 'remedy_information_requirements.csv', index=False)

verdict = {'dataset': 'nslkdd', 'alpha': ALPHA, 'focal_class': FOCAL,
           'restoration_bar': {'coverage_min': NOMINAL - TOL, 'set_size_max': SET_CAP},
           'results': res.to_dict('records'), 'E1_pass': bool(E1_PASS),
           'rungs_no_labelfree_remedy': failed_rungs,
           'restoration_by_rung': piv.to_dict(),
           'nexcp': 'not applicable to NSL-KDD (no temporal order); deferred to UGR-16',
           'entropy_note': 'entropy-matching adaptation in the spirit of ECP/EACP, '
                           'not an exact reimplementation'}
(config.REPORTS_DIR / 'remedies_verdict_nslkdd.json').write_text(json.dumps(verdict, indent=2))
print(json.dumps(verdict, indent=2))

def git(*a, show=True):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r
for s, dd in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
              ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, dd)
os.chdir(PROJECT_ROOT); git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb07: E1 remedy comparison under information constraints')
    rr = git('push','-u','origin','main')
    if rr.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-3', show=False).stdout)

{
  "dataset": "nslkdd",
  "alpha": 0.05,
  "focal_class": "R2L",
  "restoration_bar": {
    "coverage_min": 0.9299999999999999,
    "set_size_max": 2.5
  },
  "results": [
    {
      "remedy": "ACI_no_labels",
      "label_free": true,
      "rungs_restored": 0,
      "n_rungs": 5,
      "min_cov_focal": 0.03490321129210018,
      "max_set_focal": 2.1208426547352723
    },
    {
      "remedy": "EntropyAdapt",
      "label_free": true,
      "rungs_restored": 0,
      "n_rungs": 5,
      "min_cov_focal": 0.01604492114139765,
      "max_set_focal": 1.9917087087850511
    },
    {
      "remedy": "WeightedCP",
      "label_free": true,
      "rungs_restored": 0,
      "n_rungs": 5,
      "min_cov_focal": 0.16365585803924732,
      "max_set_focal": 1.6215508903705576
    },
    {
      "remedy": "ACI_with_labels",
      "label_free": false,
      "rungs_restored": 0,
      "n_rungs": 5,
      "min_cov_focal": 0.7957121551081283,
      "max_set_focal": 4.628690539874421
    },
    {
    

In [11]:
import json, subprocess, os
from pathlib import Path
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')

v = json.loads(Path('reports/gates_verdict_nslkdd.json').read_text())
g2 = v['G2_rung0_contrast']
print('G2 diff      ', round(g2['diff'], 4),      ' expect -0.8113')
print('G2 CI        ', [round(g2['ci_lo'],4), round(g2['ci_hi'],4)], ' expect [-0.8157, -0.8069]')
print('G2 pairs     ', g2['n_pairs'],             ' expect 600')
print('G2 pass      ', v['G2_pass'],              ' expect True')
print('crit3 pass   ', v['criterion3_pass'],      ' expect True')
print('crit3 gap    ', round(v['criterion3_smallest_gap'],4), ' expect 0.8113')
print('crit1 pass   ', v['criterion1_pass'],      ' expect False')
print('amend4 applies', v['amendment4_applies'],  ' expect False')
print('VIF joint    ', v['vif_joint_only'],       ' expect False')
print('corr         ', round(v['corr_scov_ssup'],4), ' expect 0.9163')

print('\nlast commit touching this file:')
print(subprocess.run(['git','log','-1','--format=%h %ad %s','--date=short',
                      '--','reports/gates_verdict_nslkdd.json'],
                     capture_output=True, text=True).stdout.strip())

G2 diff       -0.8113  expect -0.8113
G2 CI         [-0.8157, -0.8069]  expect [-0.8157, -0.8069]
G2 pairs      600  expect 600
G2 pass       True  expect True
crit3 pass    True  expect True
crit3 gap     0.8113  expect 0.8113
crit1 pass    False  expect False
amend4 applies False  expect False
VIF joint     False  expect False
corr          0.9163  expect 0.9163

last commit touching this file:
375406c 2026-07-24 nb06: gate tests and within-dataset model


In [12]:
import subprocess, os
from pathlib import Path
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')

p = Path('preregistration_amendment_05.md')
print('present:', p.exists(), '| bytes:', p.stat().st_size if p.exists() else 0)
assert p.exists(), 'drop the downloaded amendment file into the repo folder first'

def git(*a):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()); return r

git('add', 'preregistration_amendment_05.md')
git('commit', '-m', 'preregistration amendment 5: CIC-IDS2017 second-environment spec')
git('push')
git('log', '--oneline', '-3')

present: True | bytes: 5244

[main c21ce51] preregistration amendment 5: CIC-IDS2017 second-environment spec
 1 file changed, 62 insertions(+)
 create mode 100644 preregistration_amendment_05.md
To https://github.com/anasbiswas1/calshift-research.git
   9d12eca..c21ce51  main -> main
c21ce51 preregistration amendment 5: CIC-IDS2017 second-environment spec
9d12eca amendment 3: disclose placeholder-then-restore sequence
8c941b5 fix: replace amendment 3 placeholder with actual content


CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0, stdout='c21ce51 preregistration amendment 5: CIC-IDS2017 second-environment spec\n9d12eca amendment 3: disclose placeholder-then-restore sequence\n8c941b5 fix: replace amendment 3 placeholder with actual content\n', stderr='')

In [13]:
# =============================================================================
# Kaggle setup + file listing for the corrected CIC-IDS2017 mirror.
# This does NOT download yet. The mirror bundles the WTMC-2021 files we need
# with a later 2022 update we must not mix in, so we list first, then pull the
# right ones. Both steps write into Drive.
# =============================================================================
import subprocess, shutil, os
from pathlib import Path

subprocess.run(['pip', 'install', '-q', 'kaggle'], check=False)

DRIVE_ROOT = Path('/content/drive/MyDrive')
KHOME = Path('/root/.kaggle'); KHOME.mkdir(parents=True, exist_ok=True)

cands = [DRIVE_ROOT / '.kaggle' / 'kaggle.json',
         DRIVE_ROOT / 'kaggle.json',
         DRIVE_ROOT / 'CALSHIFT_Research' / 'kaggle.json']
src = next((c for c in cands if c.exists()), None)

if src is None:
    from google.colab import files
    print('Upload kaggle.json  (kaggle.com > your profile > Settings > Create New API Token)')
    up = files.upload()
    name = next(iter(up))
    (KHOME / 'kaggle.json').write_bytes(up[name])
    (DRIVE_ROOT / '.kaggle').mkdir(parents=True, exist_ok=True)
    shutil.copy(KHOME / 'kaggle.json', DRIVE_ROOT / '.kaggle' / 'kaggle.json')
    print('saved kaggle.json to Drive for next time')
else:
    shutil.copy(src, KHOME / 'kaggle.json')
    print('kaggle.json restored from', src)

os.chmod(KHOME / 'kaggle.json', 0o600)

print('\n--- files in dhoogla/distrinetcicids2017 ---')
r = subprocess.run(['kaggle', 'datasets', 'files', '-v', 'dhoogla/distrinetcicids2017'],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

Upload kaggle.json  (kaggle.com > your profile > Settings > Create New API Token)


Saving kaggle.json to kaggle.json
saved kaggle.json to Drive for next time

--- files in dhoogla/distrinetcicids2017 ---
name,size,creationDate
Benign-Monday.parquet,52413904,2023-01-11 14:37:48.867000
Bruteforce-Tuesday.parquet,43828344,2023-01-11 14:37:48.703000
DoS-Wednesday.parquet,66665039,2023-01-11 14:37:49.730000
Infiltration-Webattacks-Thursday.parquet,41223875,2023-01-11 14:37:49.410000
Portscan-DDos-Botnet-Friday.parquet,55715638,2023-01-11 14:37:50.116000



In [14]:
import subprocess
from pathlib import Path

DEST = Path('/content/drive/MyDrive/NIDS_Datasets/cicids2017-wtmc2021')
DEST.mkdir(parents=True, exist_ok=True)

subprocess.run(['kaggle', 'datasets', 'download', '-d', 'dhoogla/distrinetcicids2017',
                '-p', str(DEST), '--unzip'], check=True)

print('--- files now in the folder ---')
for p in sorted(DEST.glob('*.parquet')):
    print(f'  {p.name:45s} {p.stat().st_size/1e6:6.1f} MB')

--- files now in the folder ---
  Benign-Monday.parquet                           52.4 MB
  Bruteforce-Tuesday.parquet                      43.8 MB
  DoS-Wednesday.parquet                           66.7 MB
  Infiltration-Webattacks-Thursday.parquet        41.2 MB
  Portscan-DDos-Botnet-Friday.parquet             55.7 MB


In [15]:
# =============================================================================
# DIAGNOSTIC (read-only, no writes, no commit).
# Records the exact label/feature structure of the hashed parquet files.
# =============================================================================
import pandas as pd, numpy as np
from pathlib import Path

CIC17_DIR = Path('/content/drive/MyDrive/NIDS_Datasets/cicids2017-wtmc2021')
DAYS = ['monday','tuesday','wednesday','thursday','friday']
def day_of(f):
    low = f.lower()
    return next((d for d in DAYS if d in low), 'unknown')

frames = []
for p in sorted(CIC17_DIR.glob('*.parquet')):
    d = pd.read_parquet(p)
    d.columns = [c.strip() for c in d.columns]
    d['__day'] = day_of(p.name); d['__file'] = p.name
    frames.append(d)
cic = pd.concat(frames, ignore_index=True)

lab = next((c for c in cic.columns if c.lower() == 'label'), None)
print('rows:', len(cic), '| columns:', cic.shape[1], '| LABEL column:', lab)

print('\n=== 1. unique raw Label values and counts ===')
vc = cic[lab].astype(str).str.strip().value_counts()
for k, v in vc.items(): print(f'  {v:>9d}  {k}')
print('  n distinct labels:', len(vc))

print('\n=== 2. columns mentioning attempt / category ===')
cand = [c for c in cic.columns if 'attempt' in c.lower() or 'category' in c.lower()]
print('  candidates:', cand or 'NONE')
for c in cand:
    print(f'  {c}:', cic[c].value_counts(dropna=False).to_dict())

print('\n=== 3. attempted flows: parent recoverable from the label string? ===')
att = cic[cic[lab].astype(str).str.contains('attempt', case=False, na=False)]
print('  attempted-flagged rows:', len(att))
print('  their distinct labels:', att[lab].astype(str).str.strip().value_counts().to_dict())

print('\n=== 4. Label x Attempted Category cross-tab (if the column exists) ===')
if cand:
    print(pd.crosstab(cic[lab].astype(str).str.strip(), cic[cand[0]]).to_string())
else:
    print('  no such column; attempted info lives in the Label string only')

print('\n=== 5. scan/infiltration labels by day (are the two scans separable?) ===')
ps = cic[cic[lab].astype(str).str.contains('scan|infiltration', case=False, na=False)]
print(pd.crosstab(ps[lab].astype(str).str.strip(), ps['__day']).to_string())

print('\n=== 6. FIN/RST feature columns present ===')
finrst = [c for c in cic.columns if 'fin' in c.lower() or 'rst' in c.lower()]
print(' ', finrst or 'none found')

print('\n=== 7. full day x label cross-tab ===')
print(pd.crosstab(cic['__day'], cic[lab].astype(str).str.strip()).to_string())

rows: 1787358 | columns: 85 | LABEL column: Label

=== 1. unique raw Label values and counts ===
    1496505  Benign
     158449  DoS Hulk
      95144  DDoS
       8966  Attempted-relabel-as-Benign
       7567  DoS GoldenEye
       5485  Infiltration - Portscan
       3998  DoS Slowloris
       3972  FTP-Patator
       2961  SSH-Patator
       1741  DoS Slowhttptest
       1683  Portscan
        736  Botnet
         73  Web Attack - Brute Force
         36  Infiltration
         18  Web Attack - XSS
         13  Web Attack - SQL Injection
         11  Heartbleed
  n distinct labels: 17

=== 2. columns mentioning attempt / category ===
  candidates: NONE

=== 3. attempted flows: parent recoverable from the label string? ===
  attempted-flagged rows: 8966
  their distinct labels: {'Attempted-relabel-as-Benign': 8966}

=== 4. Label x Attempted Category cross-tab (if the column exists) ===
  no such column; attempted info lives in the Label string only

=== 5. scan/infiltration labels by d